In [373]:
import os
from unittest.mock import inplace

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from scipy import stats

In [374]:
df = pd.read_csv("../datas/hour.csv")

In [375]:
def add_rush_hour_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Giờ cao điểm đi làm / đi học (Sáng: 7h-9h, Chiều: 17h-19h vào ngày làm việc)
    df['is_rush_hour'] = (
        (df['workingday'] == 1) &
        (
            ((df['hr'] >= 7) & (df['hr'] <= 9)) |
            ((df['hr'] >= 17) & (df['hr'] <= 19))
        )
    ).astype(int)

    # 3. Giờ giải trí cuối tuần (Trưa/Chiều Thứ 7, Chủ Nhật: 10h-16h)
    df['is_weekend_leisure'] = (
        (df['workingday'] == 0) &
        (df['hr'] >= 10) & (df['hr'] <= 16)
    ).astype(int)

    return df

In [376]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.3 MB


In [377]:
target_casual = 'casual'
target_registered = 'registered'
drop_cols = [target_casual, target_registered, 'cnt',"yr","instant"]

In [378]:
df = add_rush_hour_features(df)

In [379]:
X = df.drop(columns=drop_cols)
y_casual = df[target_casual]
y_registered = df[target_registered]

In [380]:
split_idx = int(len(df) * 0.8)

In [381]:
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_cas, y_test_cas = y_casual.iloc[:split_idx], y_casual.iloc[split_idx:]
y_train_reg, y_test_reg = y_registered.iloc[:split_idx], y_registered.iloc[split_idx:]

In [382]:
def select_numerical_features(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    must_keep_cols: list = None,
) -> tuple[list[str], pd.DataFrame]:
    """
    Đánh giá và lọc Feature số tự động.
    Ưu tiên tuyệt đối: Nếu có xung đột đa cộng tuyến, biến nằm trong must_keep_cols
    sẽ luôn được giữ lại, biến còn lại bị drop dù có MI cao hơn.
    """
    must_keep_cols = set(must_keep_cols or [])
    original_features = list(X_train.columns)

    # 1. Tạo Shadow Feature để đo lường "Đáy nhiễu" (Noise Floor)
    np.random.seed(42)
    X_eval = X_train.assign(_shadow_noise_=np.random.permutation(y_train.values))

    # 2. Random Forest Importance
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_eval, y_train)
    rf_imp = dict(zip(X_eval.columns, rf.feature_importances_))

    # 3. Mutual Information
    mi_scores = mutual_info_regression(X_eval, y_train, random_state=42)
    mi_dict = dict(zip(X_eval.columns, mi_scores))

    # 4. Xác định ngưỡng động (Dynamic Thresholds)
    rf_noise_floor = rf_imp["_shadow_noise_"]
    rf_expected_value = 1.0 / len(original_features)
    rf_threshold = max(rf_noise_floor, rf_expected_value * 0.5)

    mi_noise_floor = mi_dict["_shadow_noise_"]
    real_mi_scores = [mi_dict[f] for f in original_features]
    mi_threshold = max(mi_noise_floor, np.median(real_mi_scores) * 0.5)

    # 5. Xử lý đa cộng tuyến (Redundancy với cơ chế bảo vệ must_keep_cols)
    redundancy_threshold = 0.85

    # Sắp xếp feature theo ưu tiên: Các biến must_keep lên đầu, sau đó đến MI giảm dần
    sorted_features = sorted(
        original_features,
        key=lambda x: (x not in must_keep_cols, -mi_dict[x])
    )

    corr_matrix = X_train.corr().abs()

    to_drop = set()
    kept_for_corr_check = set()

    for col in sorted_features:
        # Nếu col nằm trong must_keep_cols, ta luôn giữ và nó có quyền "đè" các biến thường khác
        is_col_redundant = False
        for kept_col in kept_for_corr_check:
            if corr_matrix.loc[col, kept_col] > redundancy_threshold:
                # Nếu col không phải must_keep nhưng xung đột với must_keep đã giữ -> drop col
                if col not in must_keep_cols:
                    is_col_redundant = True
                    break
                # Ngược lại, nếu col là must_keep mà xung đột với biến thường đã giữ trước đó,
                # ta ưu tiên drop biến thường kia đi để bảo vệ must_keep
                elif kept_col not in must_keep_cols:
                    to_drop.add(kept_col)
                    kept_for_corr_check.remove(kept_col)

        if is_col_redundant:
            to_drop.add(col)
        else:
            kept_for_corr_check.add(col)

    # 6. Tổng hợp kết quả
    results = []
    selected_features = []

    for col in original_features:
        is_must_keep = col in must_keep_cols
        is_redundant = col in to_drop
        is_good_score = (rf_imp[col] >= rf_threshold) or (mi_dict[col] >= mi_threshold)

        # Must keep luôn được chọn bất chấp redundancy hay score thấp
        is_selected = is_must_keep or (not is_redundant and is_good_score)

        if is_selected:
            selected_features.append(col)

        results.append({
            "Feature": col,
            "RF_Importance": round(rf_imp[col], 4),
            "MI_Score": round(mi_dict[col], 4),
            "Is_Redundant": is_redundant,
            "Is_Must_Keep": is_must_keep,
            "Selected": is_selected
        })

    report_df = (
        pd.DataFrame(results)
        .sort_values(by=["Selected", "RF_Importance"], ascending=[False, False])
        .reset_index(drop=True)
    )

    print(f"[INFO] Auto-Thresholds: RF >= {rf_threshold:.4f} | MI >= {mi_threshold:.4f}")

    return selected_features, report_df

In [383]:
casual_must_keep = ["atemp"]

selected_casual, report_casual = select_numerical_features(
    X_train=X_train.select_dtypes(include="number"),
    y_train=y_train_cas,
    must_keep_cols=casual_must_keep
)

[INFO] Auto-Thresholds: RF >= 0.0385 | MI >= 0.0455


In [384]:
report_casual

,Feature,RF_Importance,MI_Score,Is_Redundant,Is_Must_Keep,Selected
0,is_weekend_leisure,0.3000,0.1052,False,False,True
1,atemp,0.2124,0.2557,False,True,True
2,hr,0.1608,0.4010,False,False,True
3,hum,0.0636,0.1391,False,False,True
4,mnth,0.0362,0.1150,False,False,True
5,workingday,0.0323,0.0646,False,False,True
6,weekday,0.0180,0.0650,False,False,True
7,temp,0.0990,0.2499,True,False,False
8,windspeed,0.0254,0.0212,False,False,False
9,season,0.0106,0.0909,True,False,False


In [385]:
registered_must_keep = ["atemp"]

selected_registered, report_registered = select_numerical_features(
    X_train=X_train.select_dtypes(include="number"),
    y_train=y_train_reg,
    must_keep_cols=registered_must_keep,
)

[INFO] Auto-Thresholds: RF >= 0.0385 | MI >= 0.0252


In [386]:
report_registered

,Feature,RF_Importance,MI_Score,Is_Redundant,Is_Must_Keep,Selected
0,is_rush_hour,0.3957,0.1969,False,False,True
1,hr,0.2688,0.6614,False,False,True
2,atemp,0.0844,0.1179,False,True,True
3,hum,0.0444,0.0910,False,False,True
4,mnth,0.0196,0.0654,False,False,True
5,is_weekend_leisure,0.0171,0.0503,False,False,True
6,weekday,0.0168,0.0452,False,False,True
7,workingday,0.0064,0.0329,False,False,True
8,temp,0.0644,0.1064,True,False,False
9,windspeed,0.0231,0.0164,False,False,False


In [387]:
save_dir = "../datas"
os.makedirs(save_dir, exist_ok=True)

In [388]:
X_train[selected_casual].to_csv(f"{save_dir}/X_train_casual.csv", index=False)
X_test[selected_casual].to_csv(f"{save_dir}/X_test_casual.csv", index=False)
y_train_cas.to_csv(f"{save_dir}/y_train_casual.csv", index=False)
y_test_cas.to_csv(f"{save_dir}/y_test_casual.csv", index=False)

In [389]:
X_train[selected_registered].to_csv(f"{save_dir}/X_train_registered.csv", index=False)
X_test[selected_registered].to_csv(f"{save_dir}/X_test_registered.csv", index=False)
y_train_reg.to_csv(f"{save_dir}/y_train_registered.csv", index=False)
y_test_reg.to_csv(f"{save_dir}/y_test_registered.csv", index=False)